In [3]:
# Install dependencies
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [4]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [5]:
# Helper functions
def add_user_message(messages: list, text: str):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [6]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)
    

In [7]:
dataset = generate_dataset()
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [8]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [ ]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "test_case": test_case,
        "output": output,
        "score": score
    }

In [10]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [11]:
import json

with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Region Extraction Function\n\nHere's a Python function that extracts the AWS region from an S3 bucket URL:\n\n```python\nimport re\nfrom urllib.parse import urlparse\n\ndef extract_aws_region(s3_url: str) -> str:\n    \"\"\"\n    Extracts the AWS region from an S3 bucket URL.\n    \n    Supports formats:\n    - s3://bucket-name.region.amazonaws.com\n    - https://bucket-name.s3.region.amazonaws.com\n    - https://s3.region.amazonaws.com/bucket-name\n    - s3://bucket-name\n    \n    Args:\n        s3_url: S3 bucket URL as a string\n        \n    Returns:\n        AWS region as a string, or 'us-east-1' if region cannot be determined\n        \n    Raises:\n        ValueError: If the URL is not a valid S3 URL\n    \"\"\"\n    if not s3_url or not isinstance(s3_url, str):\n        raise ValueError(\"URL must be a non-empty string\")\n    \n    # Remove whitespace\n    s3_url = s3_url.strip()\n    \n    # Pattern 1: s3://bucket-name.region.amazonaws.com\n    p